# ML-05 — Pair Feature Vector and Leakage/Privacy Check

## 1. Build the feature vector

Counts are logged, growth is clipped to limit denominator explosions, missing growth gets flags,
and scaling will occur inside the later clustering pipeline.

In [1]:
from pathlib import Path
import duckdb, numpy as np, pandas as pd
def find_root(p=Path.cwd()):
    for x in [p,*p.parents]:
        if (x/"skills"/"README.md").exists(): return x
    raise FileNotFoundError("Run inside repository")
ROOT=find_root()
PAIR_PATH=ROOT/"work"/"outputs"/"page_pair_features.parquet"
assert PAIR_PATH.exists() and PAIR_PATH.stat().st_size>0, "Run work/scripts/build_pair_features.py"
con=duckdb.connect()
R=f"read_parquet('{PAIR_PATH.as_posix()}')"

raw=con.sql(f"""SELECT weighted_query_overlap,smaller_page_query_coverage,
smaller_page_demand_overlap,shared_query_count,shared_impression_intersection,
mean_shared_position_gap,visibility_balance,growth_a,growth_b,same_intent,
same_content_type,rare_share_a,rare_share_b,anonymized_share_a,anonymized_share_b
FROM {R}""").df()
X=pd.DataFrame(index=raw.index)
X["log_shared_queries"]=np.log1p(raw.shared_query_count)
X["log_shared_demand"]=np.log1p(raw.shared_impression_intersection)
X["weighted_query_overlap"]=raw.weighted_query_overlap.clip(0,1)
X["smaller_page_query_coverage"]=raw.smaller_page_query_coverage.clip(0,1)
X["smaller_page_demand_overlap"]=raw.smaller_page_demand_overlap.clip(0,1)
X["position_proximity"]=np.exp(-raw.mean_shared_position_gap.clip(lower=0)/10)
X["visibility_balance"]=raw.visibility_balance.clip(0,1)
X["growth_a_missing"]=raw.growth_a.isna().astype(int)
X["growth_b_missing"]=raw.growth_b.isna().astype(int)
X["growth_a"]=raw.growth_a.clip(-2,2).fillna(0)
X["growth_b"]=raw.growth_b.clip(-2,2).fillna(0)
X["movement_gap"]=(X.growth_a-X.growth_b).abs()
X["opposite_direction"]=(X.growth_a*X.growth_b<0).astype(int)
X["shared_growth"]=np.minimum(X.growth_a,X.growth_b).clip(lower=0)
X["shared_decline"]=(-np.maximum(X.growth_a,X.growth_b)).clip(lower=0)
X["same_intent"]=raw.same_intent.fillna(0).astype(int)
X["same_content_type"]=raw.same_content_type.fillna(0).astype(int)
X["hidden_query_share"]=raw[["rare_share_a","rare_share_b",
"anonymized_share_a","anonymized_share_b"]].mean(axis=1).clip(0,1)
assert not X.isna().any().any()
X.describe().T

,count,mean,std,min,25%,50%,75%,max
log_shared_queries,362562.0,1.731283,0.725043,1.098612e+00,1.098612,1.386294,2.079442,8.049746
log_shared_demand,362562.0,4.549117,1.136989,3.044522e+00,3.663562,4.304065,5.176150,11.962210
weighted_query_overlap,362562.0,0.040470,0.065273,2.125306e-05,0.005480,0.016947,0.046552,0.901024
smaller_page_query_coverage,362562.0,0.426564,0.289798,1.545595e-03,0.178571,0.375000,0.666667,1.000000
smaller_page_demand_overlap,362562.0,0.079855,0.104648,7.499863e-05,0.015795,0.042465,0.100707,0.987342
position_proximity,362562.0,0.392850,0.289478,8.980521e-15,0.129069,0.353590,0.629055,1.000000
visibility_balance,362562.0,0.407480,0.279121,2.187083e-04,0.163626,0.362775,0.626143,1.000000
growth_a_missing,362562.0,0.021679,0.145634,0.000000e+00,0.000000,0.000000,0.000000,1.000000
growth_b_missing,362562.0,0.021530,0.145143,0.000000e+00,0.000000,0.000000,0.000000,1.000000
growth_a,362562.0,-0.075480,0.777730,-1.000000e+00,-0.611111,-0.300000,0.147059,2.000000


## 2. Availability notes

All features belong to the fixed diagnostic window and exist when the queue is built. This is not
a future-outcome model. If later reframed as prediction, features must end before the target.
Hidden-query share records evidence quality, not absence of overlap.

In [2]:
pd.DataFrame([
("hashes","excluded from X","identity/grouping only"),
("last30 and prev30","allowed now","current diagnostic; no future label"),
("published/deleted","selection only","cannot teach actions"),
("product outputs","prohibited","circular decisions"),
("raw query/title/URL","not shipped","private origin fields"),
("cluster action","post-hoc only","never fed back")],
columns=["field","treatment","reason"])

,field,treatment,reason
0,hashes,excluded from X,identity/grouping only
1,last30 and prev30,allowed now,current diagnostic; no future label
2,published/deleted,selection only,cannot teach actions
3,product outputs,prohibited,circular decisions
4,raw query/title/URL,not shipped,private origin fields
5,cluster action,post-hoc only,never fed back


## 3. Automated leakage and privacy assertions

In [3]:
from pathlib import Path
import duckdb, numpy as np, pandas as pd
def find_root(p=Path.cwd()):
    for x in [p,*p.parents]:
        if (x/"skills"/"README.md").exists(): return x
    raise FileNotFoundError("Run inside repository")
ROOT=find_root()
PAIR_PATH=ROOT/"work"/"outputs"/"page_pair_features.parquet"
assert PAIR_PATH.exists() and PAIR_PATH.stat().st_size>0, "Run work/scripts/build_pair_features.py"
con=duckdb.connect()
R=f"read_parquet('{PAIR_PATH.as_posix()}')"

raw=con.sql(f"""SELECT weighted_query_overlap,smaller_page_query_coverage,
smaller_page_demand_overlap,shared_query_count,shared_impression_intersection,
mean_shared_position_gap,visibility_balance,growth_a,growth_b,same_intent,
same_content_type,rare_share_a,rare_share_b,anonymized_share_a,anonymized_share_b
FROM {R}""").df()
X=pd.DataFrame(index=raw.index)
X["log_shared_queries"]=np.log1p(raw.shared_query_count)
X["log_shared_demand"]=np.log1p(raw.shared_impression_intersection)
X["weighted_query_overlap"]=raw.weighted_query_overlap.clip(0,1)
X["smaller_page_query_coverage"]=raw.smaller_page_query_coverage.clip(0,1)
X["smaller_page_demand_overlap"]=raw.smaller_page_demand_overlap.clip(0,1)
X["position_proximity"]=np.exp(-raw.mean_shared_position_gap.clip(lower=0)/10)
X["visibility_balance"]=raw.visibility_balance.clip(0,1)
X["growth_a_missing"]=raw.growth_a.isna().astype(int)
X["growth_b_missing"]=raw.growth_b.isna().astype(int)
X["growth_a"]=raw.growth_a.clip(-2,2).fillna(0)
X["growth_b"]=raw.growth_b.clip(-2,2).fillna(0)
X["movement_gap"]=(X.growth_a-X.growth_b).abs()
X["opposite_direction"]=(X.growth_a*X.growth_b<0).astype(int)
X["shared_growth"]=np.minimum(X.growth_a,X.growth_b).clip(lower=0)
X["shared_decline"]=(-np.maximum(X.growth_a,X.growth_b)).clip(lower=0)
X["same_intent"]=raw.same_intent.fillna(0).astype(int)
X["same_content_type"]=raw.same_content_type.fillna(0).astype(int)
X["hidden_query_share"]=raw[["rare_share_a","rare_share_b",
"anonymized_share_a","anonymized_share_b"]].mean(axis=1).clip(0,1)
assert not X.isna().any().any()
X.describe().T
forbidden=("client","content_id","hash","query_id","url","title","health",
"priority","action","label")
assert not any(t in col.lower() for col in X.columns for t in forbidden)
assert np.isfinite(X.to_numpy()).all()
print(f"Checks passed: {len(X):,} vectors and {X.shape[1]} safe features.")

Checks passed: 362,562 vectors and 18 safe features.


## 4. Decision limits

IDs, product outputs, and deletion status cannot become features. Cluster action names are
interpretations, not facts. No cluster automates merging, redirecting, or deletion; intent and
business-value review remain mandatory.